In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import string
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('wordnet')
import gensim
from gensim.utils import simple_preprocess
import spacy
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ethan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ethan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ethan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
comment_df = pd.read_csv('Forum comment ferpa(in).csv', usecols = ['id', 'course_id', 'body', 'updated_at'])
thread_df = pd.read_csv('Forum thread ferpa(in).csv', usecols = ['id', 'course_id', 'body', 'updated_at'])
# json_df = pd.read_json('anonymized_ed discussion_discussion_data.json', usecols = ['text'])

In [3]:
combined_df = pd.concat([comment_df, thread_df], ignore_index=True)
# pd.merge(comment_df, thread_df, on=['id', 'course_id', 'body'], how='inner')
# combined_df.to_csv('combined_forum_with_date.csv', index=False)

In [4]:
combined_df['body'] = combined_df['body'].str.replace('PERSON', '', regex=False)
combined_df['body'] = combined_df['body'].str.lower()
# print(combined_df['body'].head())
# cleaned_comments.str.lower()
# print(cleaned_comments.head(20))
combined_df[combined_df['body'].str.count('smartbook') > 0]

,id,course_id,body,updated_at
1,\x35626565353734343336313031333039626330303137...,course-v1:GTx+CS1301xII+1T2018,smartbook that is :),11/16/2018 5:36
82,\x35633630373564323365623965363039623830303232...,course-v1:GTx+CS1301xI+1T2019,there is a link on one of the pages are you go...,2/10/2019 19:04
223,\x35626434613062613336313031333039386630303064...,course-v1:GTx+CS1301xI+3T2018,"course:cs1301xiinformation:courseguide,faqs,an...",10/27/2018 17:30
265,\x35623434663639393131336239303038346330303132...,course-v1:GTx+CS1301xI+1T2018,oh that's a good point -- that's actually stil...,7/10/2018 18:10
691,\x35386336626464306631373035363037393230303161...,course-v1:GTx+CS1301x+1T2017,i can't open chapter 1.1 of the smartbook eith...,3/13/2017 15:42
...,...,...,...,...
17218,\x35383965336139386461643636633038363230303034...,course-v1:GTx+CS1301x+1T2017,"in the smartbook practice questions, i keep fi...",2/26/2017 1:56
18303,\x35393162626137653232613866623037396630303032...,course-v1:GTx+CS1301x+1T2017,i am submitting this post to help the professo...,8/26/2017 22:03
18331,\x35386563333332656631373035363037633830303262...,course-v1:GTx+CS1301x+1T2017,smartbook is not accessable,4/11/2017 1:36
18344,\x35386464373732333962653431643038646430303237...,course-v1:GTx+CS1301x+1T2017,"if you take a a4 paper, i see only half vertic...",3/30/2017 21:22


In [5]:
def remove_punctuation(text):
    for char in text:
        if char in string.punctuation:
            text = text.replace(char, '')
    return text

In [6]:
combined_df['body'] = combined_df['body'].apply(lambda x:remove_punctuation(x))

In [7]:
stopwords_list = stopwords.words('english')
stopwords_list.extend(['ok','okay','see', 'ah', 'way', 'hi', 'hello', 'hey', 'please', 'thank', 'thanks', 'would', 'like', 'know', 'can', 'could', 'get', 'got', 'go', 'going', 'want', 'wanna', 'wont', 'will', 'wouldnt', 'should', 'shouldnt', 'im'])

In [8]:
def stopword_clean(text):
    word_list=[]
    for word in gensim.utils.simple_preprocess(text):
        if word not in stopwords_list:
            word_list.append(word)
    final_list = ' '.join(word_list)
    return final_list

In [10]:
combined_df['stopword_clean'] = combined_df['body'].apply(lambda x:stopword_clean(x))
combined_df = combined_df.dropna(subset=['stopword_clean'])
combined_df = combined_df.reset_index(drop=True)
# combined_df.to_csv('combined_forum_cleaned_with_date.csv', index=False)

In [11]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [12]:
def preprocess_text(text):
    # Tokenize
    words = word_tokenize(text.lower())  # Convert to lowercase and tokenize
    
    # Remove punctuation and non-alphabetic characters
    words = [word for word in words if word.isalpha()]
    
    # Remove stopwords
    words = [word for word in words if word not in stop_words]
    
    # Lemmatize words
    words = [lemmatizer.lemmatize(word) for word in words]
    
    # Join back into a string
    return " ".join(words)

In [13]:
combined_df['cleaned_text'] = combined_df['stopword_clean'].apply(preprocess_text)

combined_df = combined_df[combined_df['cleaned_text'].str.split().str.len().between(3, 50)]

combined_df = combined_df[['id', 'course_id', 'updated_at', 'cleaned_text']]

In [14]:
combined_df.to_csv('combined_w_date_cleaned.csv', index=False)